# 02 - Token-Aware Chunking for RAG

This notebook converts cleaned PDF pages into focused retrieval units. Chunking is not merely file splitting: it is an information-retrieval design decision that controls what evidence the embedding model can represent and what context the generator later receives.

## Learning objectives

By the end, you should be able to explain:

- Why full documents and full pages are often poor retrieval units
- Why transformer token counts matter more than word counts
- Why the embedding model's maximum sequence length constrains chunk size
- How overlap protects information near chunk boundaries
- Why we preserve source-page metadata and avoid crossing page boundaries
- How we validate chunks before embedding them

## Implementation map - where the key work happens

The notebook explains and executes the workflow. The reusable implementation is in `src/chunk_text.py`. Open that file beside this notebook and locate these stable function names:

| Concept | Exact implementation | What you need to understand |
|---|---|---|
| Hugging Face model constraint | `MODEL_NAME`, `MODEL_MAX_TOKENS` | Why the selected embedding model determines the chunk budget |
| Experiment settings | `CHUNK_CONFIGS` | Why we compare 128/24 and 220/40 rather than assume one size |
| Exact token measurement | `load_tokenizer()`, `count_tokens()` | Why model tokens, not words or characters, control truncation |
| Text units | `split_sentence_units()` | How we avoid arbitrary cuts in normal prose |
| Oversized text | `split_oversized_unit()` | How one long table or sentence is forced under budget |
| Overlap | `choose_overlap_start()` | How trailing units are repeated to protect boundary information |
| Core algorithm | `chunk_page()` | How one page becomes a list of source-aware chunks |
| Safety checks | `validate_chunks()` | How structural correctness is proved before embedding |

Line numbers may move after edits; function names are the reliable navigation points.

## Where chunking fits in RAG

```text
PDFs -> extraction and cleaning -> CHUNKING -> embeddings -> retrieval -> LLM answer
```

The embedding model produces one vector per chunk. If a chunk contains several unrelated topics, its vector becomes a blurred representation. If chunks are too small, important explanations are fragmented. We therefore compare two safe configurations instead of assuming one size is universally best.

In [1]:
from pathlib import Path
import json
import inspect
import sys

import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
sys.path.insert(0, str(project_root))

from src.chunk_text import (
    CHUNK_CONFIGS,
    MODEL_MAX_TOKENS,
    MODEL_NAME,
    build_chunk_corpus,
    chunk_page,
    count_tokens,
    load_tokenizer,
    read_jsonl,
    split_sentence_units,
    validate_chunks,
    write_jsonl,
)

pages_file = project_root / 'data' / 'processed' / 'pages.jsonl'
tokenizer_dir = project_root / 'models' / 'all-MiniLM-L6-v2-tokenizer'
pages = read_jsonl(pages_file)
tokenizer = load_tokenizer(local_tokenizer_dir=tokenizer_dir)

print('Project root:', project_root)
print('Page records loaded:', len(pages))
print('Embedding model:', MODEL_NAME)
print('Tokenizer:', type(tokenizer).__name__)
print('Enforced embedding limit:', MODEL_MAX_TOKENS, 'tokens')

Project root: <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant
Page records loaded: 159
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Tokenizer: BertTokenizerFast
Enforced embedding limit: 256 tokens


## Implementation checkpoint 1 - Hugging Face model and tokenizer

**Open in `src/chunk_text.py`:** `MODEL_NAME`, `MODEL_MAX_TOKENS`, `load_tokenizer()`, and `count_tokens()`.

**Input:** natural text. **Output:** the number of word-piece tokens the selected tokenizer will send to the model.

**Key design:** `all-MiniLM-L6-v2` maps a passage to a 384-dimensional semantic vector, but this embedding pipeline truncates input beyond 256 word-piece tokens. Therefore token-aware chunking is a correctness requirement, not just an optimization.

**You should be able to explain:** the document chunks and future user queries must be processed by the same embedding model and tokenizer so they occupy the same vector space.

## Engineering correction: tokens, not words

Our initial plan mentioned 250-word and 500-word chunks. That plan was corrected after checking the selected model. The official `all-MiniLM-L6-v2` model card states that inputs longer than 256 word pieces are truncated. Sentence Transformers also reports `model.max_seq_length = 256`. A 500-word passage would normally exceed this limit, so part of it would be silently discarded before embedding.

Sources: [Hugging Face model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) and [Sentence Transformers input-length documentation](https://sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html#input-sequence-length).

A token is a model-specific text unit. A word may become one token or several word-piece tokens. Special tokens added by the model also consume capacity. The chunker therefore measures every candidate using the exact Hugging Face tokenizer.

In [2]:
examples = [
    'Preventive maintenance follows scheduled intervals.',
    'Prognostics and Health Management supports condition-based decision-making.',
    pages[20]['cleaned_text'][:2500],
]

comparison = pd.DataFrame({
    'example': ['short sentence', 'technical sentence', 'long page excerpt'],
    'word_count': [len(text.split()) for text in examples],
    'token_count': [count_tokens(text, tokenizer) for text in examples],
})
display(comparison)
print('A word budget cannot reliably guarantee that text fits the model.')

,example,word_count,token_count
0,short sentence,5,9
1,technical sentence,7,17
2,long page excerpt,343,437


A word budget cannot reliably guarantee that text fits the model.


## Implementation checkpoint 2 - experimental configurations

**Open in `src/chunk_text.py`:** `CHUNK_CONFIGS`.

The 128-token configuration tests focused retrieval; the 220-token configuration tests richer surrounding context. Their 24-token and 40-token overlaps are about 18-19% of the target sizes.

**You should be able to explain:** these are competing hypotheses. Structural validation tells us both are legal; retrieval evaluation will determine which is better.

## Selected configurations

| Configuration | Target | Overlap | Engineering purpose |
|---|---:|---:|---|
| Focused | 128 tokens | 24 tokens | More specific vectors; close to the model's training sequence length |
| Contextual | 220 tokens | 40 tokens | More surrounding explanation while retaining margin below 256 tokens |

Both overlaps are approximately 18-19% of the target. Overlap repeats complete trailing sentence units where possible. This reduces the chance that a key definition and its explanation are separated at a boundary.

We intentionally leave headroom below 256 tokens rather than filling the absolute maximum. This makes truncation impossible under our validation rule and leaves flexibility if short prefixes are added later.

In [3]:
display(pd.DataFrame(CHUNK_CONFIGS))

,target_tokens,overlap_tokens
0,128,24
1,220,40


## Implementation checkpoint 3 - core chunking algorithm

**Open in `src/chunk_text.py`:** `split_sentence_units()`, `split_oversized_unit()`, `choose_overlap_start()`, and especially `chunk_page()`.

**Input to `chunk_page()`:** one cleaned page record, the Hugging Face tokenizer, target-token budget, and overlap-token budget.

**Output:** chunk dictionaries containing `chunk_id`, `document`, `pdf_page`, `token_count`, and `text`.

**Control flow:** prepare units -> greedily add units within budget -> save the largest valid chunk -> move backward for overlap -> repeat.

**You should be able to explain:** greedy packing retains as much complete context as possible under a hard input limit. It keeps sentence units intact where possible and token-splits only a single oversized unit.

## Chunking algorithm

For each cleaned page:

1. Split text into conservative paragraph/sentence units.
2. If one unusually long sentence or table exceeds the target, split that unit by tokenizer IDs.
3. Greedily add complete units while the candidate remains inside the token budget.
4. Finalize the chunk when the next unit would exceed the target.
5. Carry complete trailing units into the next chunk until the overlap budget is reached.
6. Preserve document, filename, PDF page, configuration, and chunk sequence metadata.

Chunks never cross a PDF-page boundary in the MVP. This makes every citation exact and easy to verify. The trade-off is that a paragraph spanning two PDF pages cannot become one chunk; overlap and retrieval of multiple chunks partly compensate for this.

In [4]:
print('Sentence-unit splitter:')
print(inspect.getsource(split_sentence_units))
print('\nPage chunker:')
print(inspect.getsource(chunk_page))

Sentence-unit splitter:
def split_sentence_units(text: str) -> list[str]:
    """Split page text into lightweight paragraph/sentence units.

    Newlines from the cleaner preserve headings and list structure. Within each
    line, a conservative regular expression splits after sentence-ending
    punctuation when the next segment appears to start a new sentence.
    """
    units: list[str] = []
    boundary = re.compile(r"(?<=[.!?])\s+(?=[A-Z0-9(\[\"“])")

    for block in re.split(r"\n+", text):
        block = re.sub(r"\s+", " ", block).strip()
        if not block:
            continue
        units.extend(part.strip() for part in boundary.split(block) if part.strip())

    return units


Page chunker:
def chunk_page(
    record: dict[str, Any],
    tokenizer: PreTrainedTokenizerBase,
    target_tokens: int,
    overlap_tokens: int,
) -> list[dict[str, Any]]:
    """Chunk one page without crossing its source-page boundary."""
    if overlap_tokens <= 0 or overlap_tokens >= target_t

## Build both chunk corpora

The same cleaned pages are processed twice. Keeping all other data constant allows chunk size to be evaluated as an independent experimental variable.

In [5]:
chunk_corpora = {}
validation_reports = []

for config in CHUNK_CONFIGS:
    target = config['target_tokens']
    overlap = config['overlap_tokens']
    chunks = build_chunk_corpus(pages, tokenizer, target, overlap)
    chunk_corpora[target] = chunks

    output_file = project_root / 'data' / 'processed' / f'chunks_{target}.jsonl'
    write_jsonl(chunks, output_file)

    report = validate_chunks(chunks, pages, target, tokenizer)
    report['target_tokens'] = target
    report['overlap_tokens'] = overlap
    validation_reports.append(report)
    print(f'Wrote {len(chunks)} chunks to {output_file}')

Wrote 972 chunks to <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant\data\processed\chunks_128.jsonl


Wrote 544 chunks to <USER_HOME>\Documents\ChatGPT\Data Analytics\engineering-document-rag-assistant\data\processed\chunks_220.jsonl


## Corpus statistics

The smaller configuration creates more chunks because each vector represents less text. More chunks can improve retrieval precision but increase embedding, storage, and search work. The larger configuration is more economical and supplies more context, but each vector may represent more than one idea.

In [6]:
statistics_rows = []
for target, chunks in chunk_corpora.items():
    frame = pd.DataFrame(chunks)
    statistics_rows.append({
        'target_tokens': target,
        'chunks': len(frame),
        'median_tokens': int(frame['token_count'].median()),
        'p95_tokens': int(frame['token_count'].quantile(0.95)),
        'maximum_tokens': int(frame['token_count'].max()),
        'median_words': int(frame['word_count'].median()),
        'pages_covered': frame[['document', 'pdf_page']].drop_duplicates().shape[0],
    })

statistics_df = pd.DataFrame(statistics_rows)
display(statistics_df)

,target_tokens,chunks,median_tokens,p95_tokens,maximum_tokens,median_words,pages_covered
0,128,972,114,128,128,76,159
1,220,544,198,219,220,129,159


## Inspect adjacent chunks and overlap

The example below prints adjacent chunks from one page. Repeated trailing text at the start of the next chunk is intentional overlap, not accidental duplication. The overlap setting is a desired budget for complete units, not a guarantee that every boundary will have exactly that number. If no complete unit fits, the implementation may carry one trailing unit up to twice the desired budget; a single very long unit can still require zero overlap to guarantee forward progress.

In [7]:
example_chunks = [
    chunk for chunk in chunk_corpora[128]
    if chunk['document'] == 'NIST.AMS.100-34' and chunk['pdf_page'] == 16
]

for chunk in example_chunks[:3]:
    print('=' * 100)
    print(chunk['chunk_id'], '| tokens:', chunk['token_count'], '| words:', chunk['word_count'])
    print('-' * 100)
    print(chunk['text'])
    print()

NIST.AMS.100-34-p016-t128-c001 | tokens: 110 | words: 95
----------------------------------------------------------------------------------------------------
necessary PM, this excess can lead to wasted money in unnecessary labor hours and materials.
PM is planned downtime which is still downtime.
The more time a piece of equipment is down, the less opportunity there is for it to be operating to produce a product or perform a process to support the organization’s revenue stream.
An organization typically designs the PM strategy to be a tradeoff between the potential for excess maintenance and the risk of not doing enoug h maintenance thereby increasing the presence of a fault and failure lead ing to RM.

NIST.AMS.100-34-p016-t128-c002 | tokens: 110 | words: 86
----------------------------------------------------------------------------------------------------
An organization typically designs the PM strategy to be a tradeoff between the potential for excess maintenance and the risk of 

## Implementation checkpoint 4 - validation

**Open in `src/chunk_text.py`:** `validate_chunks()`.

It checks unique IDs, non-empty chunks, token-budget compliance, complete page coverage, valid page references, and measured suffix/prefix overlap between adjacent chunks.

**You should be able to explain:** validation answers `Did we construct the chunk dataset correctly?`; later retrieval evaluation answers `Does it retrieve the correct evidence?` These are different engineering questions.

## Validation invariants

Before embedding, the chunk corpus must satisfy non-negotiable invariants:

- Every chunk ID is unique.
- Every chunk contains text.
- No chunk exceeds its target token budget.
- Every retained source page produces at least one chunk.
- No chunk points to an unknown source page.
- Actual adjacent overlap is measured rather than inferred from configuration.

In [8]:
validation_df = pd.DataFrame(validation_reports).set_index('target_tokens')
display(validation_df)

failure_columns = [
    'duplicate_chunk_ids', 'empty_chunks', 'over_budget_chunks',
    'missing_source_pages', 'unknown_source_pages',
]
assert (validation_df[failure_columns] == 0).all().all()
assert validation_df.index.max() < MODEL_MAX_TOKENS
print('All chunking validation checks passed. The corpora are ready for embedding.')

,chunks,duplicate_chunk_ids,empty_chunks,over_budget_chunks,missing_source_pages,unknown_source_pages,adjacent_same_page_pairs,zero_overlap_pairs,median_actual_overlap_tokens,maximum_actual_overlap_tokens,overlap_tokens
target_tokens,,,,,,,,,,,
128,972,0,0,0,0,0,813,74,18,46,24
220,544,0,0,0,0,0,385,10,28,75,40


All chunking validation checks passed. The corpora are ready for embedding.


## Engineering interpretation

We have not yet proven that 128 or 220 tokens retrieves better answers. Chunking validation proves structural correctness, not retrieval quality. Both corpora will be embedded using the same model and evaluated using the same questions. Retrieval metrics will determine the preferred configuration.

## Interview-ready explanation

> I designed a token-aware chunking pipeline around the embedding model's 256-token input constraint. I corrected an initial word-based plan after identifying that long passages would be silently truncated. I compared 128-token and 220-token chunks with approximately 18-19% overlap, preserved exact PDF-page metadata for citations, kept sentence units intact where possible, and validated that no chunks exceeded the model budget or lost source coverage. I then treated chunk size as an experimental variable for retrieval evaluation rather than assuming a universal optimum.

## Next stage

Load `all-MiniLM-L6-v2`, embed both chunk corpora into 384-dimensional vectors, implement cosine-similarity search, and compare retrieval results using identical test questions.